In [0]:
# 01 - Ler a tabela Bronze

from pyspark.sql import functions as F

df_bronze = spark.table(
    "workspace.projeto_integrador.bronze_exames_citopatologicos"
)

display(df_bronze.limit(10))

In [0]:
# 02 - Criar a camada Silver

df_silver = (
    df_bronze

    .filter(F.col("sg_granularidade") == "MN")

    .select(
        F.col("co_anomes").cast("string").alias("ano_mes"),
        F.col("co_ibge").cast("string").alias("codigo_ibge"),
        F.col("no_municipio").alias("municipio"),
        F.col("co_uf").cast("string").alias("codigo_uf"),
        F.col("sg_uf").alias("uf"),
        F.col("no_uf").alias("estado"),

        F.col("co_regiao_brasil")
            .cast("string")
            .alias("codigo_regiao_brasil"),

        F.col("no_regiao_brasil").alias("regiao_brasil"),
        F.col("sg_regiao_brasil").alias("sigla_regiao_brasil"),

        F.col("co_regiao_saude")
            .cast("string")
            .alias("codigo_regiao_saude"),

        F.col("no_regiao_saude").alias("regiao_saude"),

        F.col("co_macro")
            .cast("string")
            .alias("codigo_macrorregiao"),

        F.col("no_macro").alias("macrorregiao"),

        F.col("vl_indicador_calculado_mun")
            .cast("double")
            .alias("valor_indicador_municipio"),

        F.col("dt_competencia").alias("data_competencia_original"),
        F.col("dt_atualizacao").alias("data_atualizacao_original"),
        F.col("ds_unidade_medida").alias("unidade_medida"),
        F.col("sg_granularidade").alias("sigla_granularidade"),
        F.col("ds_granularidade").alias("granularidade")
    )

    .withColumn("municipio", F.trim(F.col("municipio")))
    .withColumn("uf", F.trim(F.col("uf")))
    .withColumn("estado", F.trim(F.col("estado")))
    .withColumn("regiao_brasil", F.trim(F.col("regiao_brasil")))
    .withColumn("regiao_saude", F.trim(F.col("regiao_saude")))
    .withColumn("macrorregiao", F.trim(F.col("macrorregiao")))

    .withColumn(
        "ano",
        F.substring(F.col("ano_mes"), 1, 4).cast("int")
    )
    .withColumn(
        "mes",
        F.substring(F.col("ano_mes"), 5, 2).cast("int")
    )

    .withColumn(
        "competencia",
        F.to_date(
            F.concat(
                F.substring(F.col("ano_mes"), 1, 4),
                F.lit("-"),
                F.substring(F.col("ano_mes"), 5, 2),
                F.lit("-01")
            )
        )
    )
)

display(df_silver.limit(10))

In [0]:
# 03 - Conferir a estrutura da Silver

df_silver.printSchema()

print("Quantidade de registros:", df_silver.count())
print("Quantidade de colunas:", len(df_silver.columns))

print(
    "Quantidade de municípios:",
    df_silver.select("codigo_ibge").distinct().count()
)

print(
    "Quantidade de competências:",
    df_silver.select("ano_mes").distinct().count()
)

df_silver.select(
    F.min("competencia").alias("primeira_competencia"),
    F.max("competencia").alias("ultima_competencia")
).show()


In [0]:
# 04 - Verificar duplicidades por município e competência

duplicados = (
    df_silver
    .groupBy("ano_mes", "codigo_ibge")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "Quantidade de combinações município/competência duplicadas:",
    duplicados.count()
)

display(duplicados.limit(20))


In [0]:
# 05 - Salvar a tabela Silver

df_silver.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(
        "workspace.projeto_integrador.silver_exames_citopatologicos"
    )

In [0]:
# 06 - Validar a tabela Silver criada

df_silver_salva = spark.table(
    "workspace.projeto_integrador.silver_exames_citopatologicos"
)

display(df_silver_salva.limit(10))

In [0]:
# 07 - Resumo da base Silver

resumo = df_silver.select(
    F.count("*").alias("total_registros"),
    F.countDistinct("codigo_ibge").alias("total_municipios"),
    F.countDistinct("ano_mes").alias("total_competencias"),
    F.min("competencia").alias("primeira_competencia"),
    F.max("competencia").alias("ultima_competencia"),
    F.min("ano").alias("primeiro_ano"),
    F.max("ano").alias("ultimo_ano")
)

display(resumo)

In [0]:
# 08 - Verificar quantidade de registros por ano

registros_por_ano = (
    df_silver
    .groupBy("ano")
    .agg(
        F.count("*").alias("quantidade_registros"),
        F.countDistinct("codigo_ibge").alias("quantidade_municipios")
    )
    .orderBy("ano")
)

display(registros_por_ano)

In [0]:
# 09 - Verificar todas as competências disponíveis

competencias = (
    df_silver
    .select("ano_mes", "competencia")
    .distinct()
    .orderBy("competencia")
)

display(competencias)

In [0]:
# 10 - Verificar valores zero do indicador por ano

analise_zeros = (
    df_silver
    .groupBy("ano")
    .agg(
        F.count("*").alias("total_registros"),
        F.sum(
            F.when(F.col("valor_indicador_municipio") == 0, 1).otherwise(0)
        ).alias("quantidade_zeros"),
        F.sum(
            F.when(F.col("valor_indicador_municipio") > 0, 1).otherwise(0)
        ).alias("quantidade_maior_zero")
    )
    .withColumn(
        "percentual_zeros",
        F.round(
            (F.col("quantidade_zeros") / F.col("total_registros")) * 100,
            2
        )
    )
    .orderBy("ano")
)

display(analise_zeros)

In [0]:
# 11 - Estatísticas do indicador por ano

estatisticas_anuais = (
    df_silver
    .groupBy("ano")
    .agg(
        F.min("valor_indicador_municipio").alias("valor_minimo"),
        F.max("valor_indicador_municipio").alias("valor_maximo"),
        F.round(
            F.avg("valor_indicador_municipio"),
            2
        ).alias("valor_medio"),
        F.expr(
            "percentile_approx(valor_indicador_municipio, 0.5)"
        ).alias("mediana"),
        F.sum("valor_indicador_municipio").alias("total_indicador")
    )
    .orderBy("ano")
)

display(estatisticas_anuais)

In [0]:
# 12 - Verificar quantos municípios já tiveram pelo menos um valor maior que zero

municipios_com_exames = (
    df_silver
    .filter(
        (F.col("ano") <= 2025) &
        (F.col("valor_indicador_municipio") > 0)
    )
    .select(
        "codigo_ibge",
        "municipio",
        "uf"
    )
    .distinct()
)

print(
    "Municípios que tiveram valor maior que zero entre 2016 e 2025:",
    municipios_com_exames.count()
)

print(
    "Total de municípios existentes na base:",
    df_silver
    .filter(F.col("ano") <= 2025)
    .select("codigo_ibge")
    .distinct()
    .count()
)

In [0]:
# 13 - Verificar em quantos anos cada município teve valor maior que zero

frequencia_municipios = (
    df_silver
    .filter(F.col("ano") <= 2025)
    .groupBy(
        "codigo_ibge",
        "municipio",
        "uf"
    )
    .agg(
        F.sum(
            F.when(
                F.col("valor_indicador_municipio") > 0,
                1
            ).otherwise(0)
        ).alias("anos_com_valor_maior_zero"),

        F.count("*").alias("anos_na_base")
    )
)

resumo_frequencia = (
    frequencia_municipios
    .groupBy("anos_com_valor_maior_zero")
    .agg(
        F.count("*").alias("quantidade_municipios")
    )
    .orderBy("anos_com_valor_maior_zero")
)

display(resumo_frequencia)

In [0]:
# 14 - Municípios com valores positivos em todos os anos completos

municipios_recorrentes = (
    frequencia_municipios
    .filter(
        F.col("anos_com_valor_maior_zero") == 10
    )
    .orderBy("uf", "municipio")
)

print(
    "Municípios com valor maior que zero em todos os anos de 2016 a 2025:",
    municipios_recorrentes.count()
)

display(municipios_recorrentes.limit(50))